# 5.2 Coding Practice: Text Classification, Evaluation, and Error Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module5/week5_coding_practice.ipynb)

This formative notebook follows one workflow: **text → representation → classifier → probability → prediction → evaluation → error analysis**. Work through it in order. Before each interpretation checkpoint, make a short prediction or note in your own words; later cells do not depend on your written response.

By the end, you should be able to explain why two classifiers can make different decisions on the same text and why a high metric alone is not enough to trust a system.

## Learning goals

- Read a confusion matrix, accuracy, precision, recall, and F1 as evidence about model behavior.
- Train a TF-IDF + logistic-regression baseline and inspect its probabilities and errors.
- Compare sparse lexical features with dense sentence embeddings.
- Treat an LLM classifier as another decision system to evaluate—not an automatic winner.

**Course bridge:** similarity asks which texts are close. Classification uses labeled examples to decide which predefined category a new text should receive.

In [ ]:
import importlib.util
import subprocess
import sys

# One-time setup for the optional dense-embedding comparison.
# This is the only package installation in the core activity; Colab does not
# include sentence-transformers by default. It is safe to rerun.
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"] )


In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split

pd.set_option("display.max_colwidth", 100)
RANDOM_STATE = 42


## 1. Metrics describe mistakes, not just success

Imagine a system that flags potentially dangerous support requests for urgent review. `security` is the positive class below. The rows are deliberately imbalanced: most ordinary requests are not security issues.

In [ ]:
metric_examples = pd.DataFrame({
    "ticket": [
        "A suspicious email asked me to verify my password.",
        "My laptop shows a ransom message after opening an attachment.",
        "How do I connect my visitor to Wi-Fi?",
        "My print job is waiting at the release station.",
        "I forgot my account password.",
        "The classroom projector is blank.",
        "My wireless connection drops in the library.",
        "Can I add money to my print account?",
        "I need help sharing a folder with my project team.",
        "My account is locked after too many sign-in attempts.",
    ],
    "true_label": ["security", "security"] + ["not_security"] * 8,
    "predicted_label": ["security", "not_security", "security", "not_security", "not_security", "not_security", "not_security", "not_security", "not_security", "not_security"],
})
metric_examples


In [ ]:
y_true = metric_examples["true_label"]
y_pred = metric_examples["predicted_label"]
metric_summary = pd.Series({
    "accuracy": accuracy_score(y_true, y_pred),
    "precision (security)": precision_score(y_true, y_pred, pos_label="security"),
    "recall (security)": recall_score(y_true, y_pred, pos_label="security"),
    "F1 (security)": f1_score(y_true, y_pred, pos_label="security"),
}).round(2)
display(metric_summary.to_frame("value"))
cm = confusion_matrix(y_true, y_pred, labels=["security", "not_security"])
ConfusionMatrixDisplay(cm, display_labels=["security", "not_security"]).plot()
print("Rows with a mistake:")
display(metric_examples.loc[y_true != y_pred])
assert cm.tolist() == [[1, 1], [1, 7]]


### Pause and interpret

1. Which error occurs here more often: a false positive or a false negative? Point to the rows.
2. Why can the accuracy look reassuring even though the system misses a true security request?
3. For an urgent-security review queue, would you prioritize precision or recall? What cost are you accepting?
4. What would change if there were 1,000 ordinary tickets and still only two security tickets?

## 2. Build a transparent text-classification baseline

Our small fictional help desk has four labels. Each text is intentionally short enough to read. Some test examples are ambiguous, use negation, or use wording that does not closely match the training text. In a real project, the corpus would be much larger; here, interpretability is the point.

In [ ]:
tickets = pd.DataFrame([
    ("account_access", "I forgot my password and cannot enter my account."),
    ("account_access", "My sign-in is locked after several failed attempts."),
    ("account_access", "I bought a new phone and cannot approve my login."),
    ("account_access", "Where can I find my university username?"),
    ("account_access", "Please reset the password for my email account."),
    ("account_access", "The authentication app is on my old phone."),
    ("network_access", "My laptop cannot join the secure campus Wi-Fi."),
    ("network_access", "A visitor needs the guest wireless network."),
    ("network_access", "The internet disconnects every few minutes in the library."),
    ("network_access", "How do I register a game console for wireless access?"),
    ("network_access", "The residence hall Ethernet jack does not work."),
    ("network_access", "My computer keeps choosing the wrong campus network."),
    ("printing", "How do I add the campus printer to my laptop?"),
    ("printing", "My document is in the queue but has not printed."),
    ("printing", "Where do I release a print job?"),
    ("printing", "The printer says it is offline and shows a paper jam."),
    ("printing", "I need to check my remaining printing credit."),
    ("printing", "The print queue sent my pages to the wrong device."),
    ("security", "I received a phishing email asking for my password."),
    ("security", "My laptop may have malware after opening a strange file."),
    ("security", "My phone was stolen and it has my sign-in approvals."),
    ("security", "Where do I locate an encryption recovery key?"),
    ("security", "A message asked for urgent payment and account verification."),
    ("security", "I clicked a suspicious attachment; what should I do?"),
], columns=["label", "text"])
tickets.groupby("label").size().to_frame("examples")


### Predict before training

Read a few rows. Which words seem like useful evidence? Which texts could plausibly receive two labels? Do not expect a classifier to know intent that is not represented in its labeled examples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    tickets["text"], tickets["label"], test_size=0.50,
    random_state=RANDOM_STATE, stratify=tickets["label"]
)
print(f"Training examples: {len(X_train)}; evaluation examples: {len(X_test)}")
print("Evaluation labels:", y_test.value_counts().sort_index().to_dict())

tfidf = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)
tfidf_classifier = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
tfidf_classifier.fit(X_train_tfidf, y_train)
print("TF-IDF matrix shape:", X_train_tfidf.shape)
print("A few learned features:", tfidf.get_feature_names_out()[:12].tolist())
assert X_train_tfidf.shape[1] == X_test_tfidf.shape[1]


In [ ]:
def prediction_table(model, features, texts, true_labels, approach):
    probabilities = model.predict_proba(features)
    predicted = model.classes_[probabilities.argmax(axis=1)]
    table = pd.DataFrame({
        "text": list(texts),
        "true_label": list(true_labels),
        "predicted_label": predicted,
        "confidence": probabilities.max(axis=1),
        "approach": approach,
    })
    for index, label in enumerate(model.classes_):
        table[f"P({label})"] = probabilities[:, index]
    return table.sort_values("confidence", ascending=False).reset_index(drop=True)

tfidf_results = prediction_table(
    tfidf_classifier, X_test_tfidf, X_test, y_test, "TF-IDF + logistic regression"
)
display(tfidf_results.round(3))
print(classification_report(y_test, tfidf_results["predicted_label"], zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, tfidf_results["predicted_label"], xticks_rotation=45)


In [ ]:
# Logistic-regression feature weights are one kind of evidence, not a complete explanation.
feature_names = tfidf.get_feature_names_out()
top_features = []
for label, weights in zip(tfidf_classifier.classes_, tfidf_classifier.coef_):
    strongest = feature_names[np.argsort(weights)[-5:]][::-1]
    top_features.append({"label": label, "strongest positive features": ", ".join(strongest)})
pd.DataFrame(top_features)


### Error-analysis checkpoint

Run the next cell and inspect at least two rows. For each, state: (a) what lexical evidence may have helped or hurt, (b) whether the label definition or the text is ambiguous, and (c) what additional training examples might help. A confidence score is a model score, not proof that the label is correct.

In [ ]:
tfidf_errors = tfidf_results.query("true_label != predicted_label")
display(tfidf_errors if not tfidf_errors.empty else pd.DataFrame({"note": ["No errors in this split—inspect low-confidence rows instead."]}))
display(tfidf_results.sort_values("confidence").head(4).round(3))


### Optional threshold observation

A multiclass model usually chooses the largest class probability. To connect with the binary threshold discussion, temporarily ask whether a ticket should be escalated as `security` when its security probability exceeds a chosen cutoff. This is an observation, not threshold optimization.

In [ ]:
SECURITY_THRESHOLD = 0.25  # Try 0.10, 0.25, and 0.50.
security_probabilities = tfidf_results["P(security)"]
security_true = y_test.eq("security")
security_predicted = security_probabilities.ge(SECURITY_THRESHOLD)
threshold_metrics = pd.Series({
    "precision": precision_score(security_true, security_predicted, zero_division=0),
    "recall": recall_score(security_true, security_predicted, zero_division=0),
    "false positives": int((~security_true & security_predicted).sum()),
    "false negatives": int((security_true & ~security_predicted).sum()),
})
display(threshold_metrics.to_frame(f"threshold = {SECURITY_THRESHOLD}"))
assert security_probabilities.between(0, 1).all()


## 3. Change the representation, keep the classifier

Now the pipeline is **text → sentence embedding → logistic regression**. A sentence embedding can capture some semantic similarity beyond exact word overlap. It does not make labels, training data, or error analysis unnecessary. The first run downloads the small public `all-MiniLM-L6-v2` model; this requires an internet connection.

In [ ]:
from sentence_transformers import SentenceTransformer

try:
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    X_train_embeddings = embedding_model.encode(X_train.tolist(), show_progress_bar=False)
    X_test_embeddings = embedding_model.encode(X_test.tolist(), show_progress_bar=False)
    embedding_classifier = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    embedding_classifier.fit(X_train_embeddings, y_train)
    embedding_results = prediction_table(
        embedding_classifier, X_test_embeddings, X_test, y_test,
        "sentence embeddings + logistic regression"
    )
    print("Embedding matrix shape:", X_train_embeddings.shape)
    display(embedding_results.round(3))
    assert X_train_embeddings.shape[1] == X_test_embeddings.shape[1]
    assert np.isfinite(X_train_embeddings).all()
except Exception as embedding_error:
    embedding_results = None
    print("Dense comparison skipped because the model could not be loaded.")
    print(f"Reason: {embedding_error}")
    print("In Colab, confirm internet access and rerun this cell; Parts 1–2 remain valid offline.")


In [ ]:
if embedding_results is not None:
    comparison = (
        tfidf_results[["text", "true_label", "predicted_label", "confidence"]]
        .rename(columns={"predicted_label": "tfidf_prediction", "confidence": "tfidf_confidence"})
        .merge(
            embedding_results[["text", "predicted_label", "confidence"]]
            .rename(columns={"predicted_label": "embedding_prediction", "confidence": "embedding_confidence"}),
            on="text",
        )
    )
    comparison["agree"] = comparison["tfidf_prediction"] == comparison["embedding_prediction"]
    display(comparison.sort_values(["agree", "text"]).round(3))
    print("TF-IDF accuracy:", round(accuracy_score(y_test, tfidf_results["predicted_label"]), 3))
    print("Embedding accuracy:", round(accuracy_score(y_test, embedding_results["predicted_label"]), 3))
else:
    comparison = None
    print("Representation comparison is waiting for the sentence-embedding model.")


### Compare representations

Find one row where the two representations disagree (or one low-confidence row if they all agree). Why might TF-IDF favor lexical overlap while embeddings favor broader semantic similarity? Which prediction is better supported by the text and the label definitions? Do not answer only with an accuracy number.

## 4. Optional Gemini classification comparison

This section is independent of Parts 1–3. Skip it if you do not have an API key, encounter quota limits, or prefer not to make an API call. It sends only the 12 fictional evaluation tickets—not personal, sensitive, or course-submission data—to Gemini.

### Setup before any Gemini code

1. Go to [Google AI Studio](https://aistudio.google.com/) and create or retrieve an API key.
2. In Colab, open the key icon (**Secrets**), add a secret named `GEMINI_API_KEY`, paste the value there, and enable notebook access.
3. **Never paste, print, submit, or commit an API key.**
4. In the configuration cell, choose a currently available lightweight **Flash** text model shown in AI Studio. Model availability changes, so this notebook intentionally does not hard-code a model name.

Troubleshooting: a missing-key message means the Secret is absent or notebook access is disabled; a quota/rate-limit message means wait or reduce requests; an import error means rerun the SDK installation cell and restart the runtime if Colab asks. The official SDK is `google-genai`.

In [ ]:
# Optional Gemini setup. This runs only if you choose to configure the section.
RUN_GEMINI = False  # Change to True only after adding GEMINI_API_KEY in Colab Secrets.
GEMINI_MODEL = "YOUR_CURRENT_FLASH_MODEL"  # Copy a current Flash text-model ID from AI Studio.

if RUN_GEMINI:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "google-genai"])
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    if not GEMINI_API_KEY:
        raise RuntimeError("Add GEMINI_API_KEY to Colab Secrets and enable notebook access; do not paste it into this notebook.")
    if GEMINI_MODEL == "YOUR_CURRENT_FLASH_MODEL":
        raise ValueError("Choose a current Flash text-model ID in GEMINI_MODEL before running.")
else:
    print("Gemini section skipped. Core classification work is complete without an API key.")


In [ ]:
# Minimal connection test. It makes no classification calls.
if RUN_GEMINI:
    from google import genai
    client = genai.Client(api_key=GEMINI_API_KEY)
    test_response = client.models.generate_content(
        model=GEMINI_MODEL, contents="Reply with exactly: connected"
    )
    print(test_response.text.strip())
else:
    print("Connection test skipped.")


In [ ]:
LABEL_DEFINITIONS = {
    "account_access": "passwords, usernames, sign-in locks, or authentication devices",
    "network_access": "Wi-Fi, Ethernet, guest network, or device registration",
    "printing": "printers, print queues, release stations, or print credit",
    "security": "phishing, malware, stolen devices, suspicious links, or recovery keys",
}

def gemini_label(text):
    label_lines = "\n".join(f"- {label}: {definition}" for label, definition in LABEL_DEFINITIONS.items())
    prompt = f"""Classify this fictional support ticket using exactly one label.
Labels:
{label_lines}
Ticket: {text}
Return only one label from: {', '.join(LABEL_DEFINITIONS)}."""
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    label = response.text.strip().lower().replace("`", "")
    return label if label in LABEL_DEFINITIONS else "invalid_response"

if RUN_GEMINI:
    # The held-out evaluation split has 12 selected examples, keeping usage small.
    gemini_results = pd.DataFrame({"text": X_test.tolist(), "true_label": y_test.tolist()})
    gemini_results["gemini_prediction"] = gemini_results["text"].apply(gemini_label)
    display(gemini_results)
else:
    gemini_results = None
    print("Classification calls skipped.")


### Gemini comparison checkpoint

If you ran the optional section, compare the same rows. Find one agreement, one disagreement, and one ambiguous example. Which decision is better supported by the label definitions? Optionally revise **one** prompt phrase (not the labels) and observe whether the prediction changes. A changing response is evidence that the prompt is part of the decision process.

## Final model comparison and takeaway

Complete the last column with observations from your own run. Do not choose a winner from a single numerical score. Consider data needs, errors, latency/cost, privacy, interpretability, and what the application treats as harmful.

| Approach | Representation | Learns from this dataset? | Key observation |
|---|---|---|---|
| TF-IDF + Logistic Regression | Sparse lexical vectors | Yes |  |
| Sentence Embeddings + Logistic Regression | Dense semantic vectors | Yes |  |
| Gemini | LLM internal representation; prompted label definitions | No (zero-shot) |  |

**Final check:** identify one prediction that surprised you, explain likely evidence or ambiguity behind it, and name the false positive or false negative that would matter most for this help desk. Text classification is not just calling a model: we choose a representation, define labels, evaluate decisions, and inspect errors before deciding whether a system is useful.

In [ ]:
# Transparent checks for the core workflow. These validate computation, not usefulness.
assert len(tickets) == 24
assert set(tickets["label"]) == set(tfidf_classifier.classes_)
assert len(tfidf_results) == len(y_test) == 12
assert tfidf_results.filter(regex=r"^P\(").sum(axis=1).round(8).eq(1).all()
if embedding_results is not None:
    assert set(embedding_results["predicted_label"]).issubset(set(tickets["label"]))
print("Core workflow checks passed.")
